In [1]:
from collections import defaultdict
from itertools import combinations
import re


LARGE_QASM_BYTES = 40_000_000
SAMPLE_CHUNK_BYTES = 1_000_000


_IDENT = r"[A-Za-z_][A-Za-z0-9_]*"
_QREG2_RE = re.compile(
    rf"^qreg\s+({_IDENT})\s*\[\s*(\d+)\s*\]\s*$", re.IGNORECASE
)
_QUBIT3_ARRAY_RE = re.compile(
    rf"^qubit\s*\[\s*(\d+)\s*\]\s*({_IDENT})\s*$", re.IGNORECASE
)
_QUBIT3_ALT_RE = re.compile(
    rf"^qubit\s+({_IDENT})\s*\[\s*(\d+)\s*\]\s*$", re.IGNORECASE
)
_QUBIT3_SCALAR_RE = re.compile(rf"^qubit\s+({_IDENT})\s*$", re.IGNORECASE)
_CREG2_RE = re.compile(
    rf"^creg\s+({_IDENT})\s*\[\s*(\d+)\s*\]\s*$", re.IGNORECASE
)
_BIT3_ARRAY_RE = re.compile(
    rf"^bit\s*\[\s*(\d+)\s*\]\s*({_IDENT})\s*$", re.IGNORECASE
)
_BIT3_SCALAR_RE = re.compile(rf"^bit\s+({_IDENT})\s*$", re.IGNORECASE)
_OP_RE = re.compile(
    rf"^({_IDENT})\s*(\([^;]*\))?\s+(.+)$", re.IGNORECASE
)
_REF_RE = re.compile(
    rf"^({_IDENT})(?:\s*\[\s*(\d+)(?:\s*:\s*(\d+))?\s*\])?$"
)
_HARDWARE_QUBIT_RE = re.compile(r"^\$(\d+)$")
_LEADING_IF_RE = re.compile(r"^if\s*\([^)]*\)\s*", re.IGNORECASE)
_DECLARATION_LINE_RE = re.compile(
    r"(?m)^(?:qubit|qreg|bit|creg)\b", re.IGNORECASE
)
_MCPHASE_VARIANT_RE = re.compile(r"^mcphase(?:_\d+)?$")
_GENERATED_CIRCUIT_GATE_RE = re.compile(
    r"^_c?circuit_\d+(?:_\d+)?(?:_dg)?$"
)

# These fields are useful for diagnostics but are not recommended as initial
# simulator-runtime inputs. They can introduce format, classical-register, or
# parser-strategy signals that may not generalize to hidden circuits.
MODEL_EXCLUDED_FEATURES = frozenset({
    "qasm_version",
    "n_clbits",
    "measurement_count",
    "barrier_count",
    "delay_count",
    "sampled",
    "sample_fraction",
    "definitions_skipped",
    "source_fraction",
})


_SKIP_PREFIXES = (
    "openqasm",
    "include",
    "input",
    "output",
    "const",
    "let",
    "int",
    "uint",
    "float",
    "angle",
    "bool",
    "duration",
    "stretch",
    "extern",
    "pragma",
)

_CLIFFORD_GATES = {
    "id", "i", "x", "y", "z", "h", "s", "sdg", "sx", "sxdg",
    "cx", "cnot", "cy", "cz", "ch", "swap", "ecr",
}
_DIAGONAL_GATES = {
    "z", "s", "sdg", "t", "tdg", "p", "u1", "rz", "cz", "cp",
    "cu1", "rzz", "ccz",
}
_ROTATION_GATES = {
    "p", "u", "u1", "u2", "u3", "rx", "ry", "rz", "rxx", "ryy",
    "rzz", "rzx", "cp", "crx", "cry", "crz", "cu", "cu1", "cu2",
    "cu3", "xx_plus_yy", "xx_minus_yy",
}


def _strip_comments(line, in_block_comment):
    """Remove // and /* */ comments from one line without copying the file."""
    if not in_block_comment and "/" not in line:
        return line, False
    pieces = []
    pos = 0
    size = len(line)
    while pos < size:
        if in_block_comment:
            end = line.find("*/", pos)
            if end < 0:
                return "".join(pieces), True
            pos = end + 2
            in_block_comment = False
            continue

        line_comment = line.find("//", pos)
        block_comment = line.find("/*", pos)
        if line_comment >= 0 and (block_comment < 0 or line_comment < block_comment):
            pieces.append(line[pos:line_comment])
            break
        if block_comment < 0:
            pieces.append(line[pos:])
            break
        pieces.append(line[pos:block_comment])
        pos = block_comment + 2
        in_block_comment = True
    return "".join(pieces), in_block_comment


def _iter_statements(qasm_text):
    """Yield semicolon-terminated statements while skipping gate definitions."""
    pending = ""
    in_block_comment = False
    definition_depth = 0
    waiting_for_definition_body = False

    # splitlines() would allocate a second, very large list for a 200 MB file.
    start = 0
    text_size = len(qasm_text)
    while start < text_size:
        end = qasm_text.find("\n", start)
        if end < 0:
            end = text_size
        raw_line = qasm_text[start:end]
        start = end + 1

        line, in_block_comment = _strip_comments(raw_line, in_block_comment)
        stripped = line.strip()
        if not stripped:
            continue

        if definition_depth:
            definition_depth += stripped.count("{") - stripped.count("}")
            if definition_depth <= 0:
                definition_depth = 0
            continue

        if waiting_for_definition_body:
            if "{" in stripped:
                definition_depth = stripped.count("{") - stripped.count("}")
                waiting_for_definition_body = False
            continue

        # Definitions are rare. Avoid split/lower allocation on every gate.
        lowered_start = stripped[:8].lower()
        first_word = lowered_start.split(None, 1)[0]
        if first_word in {"gate", "def", "defcal"}:
            pending = ""
            if "{" in stripped:
                definition_depth = stripped.count("{") - stripped.count("}")
            elif first_word != "gate" or not stripped.endswith(";"):
                waiting_for_definition_body = True
            continue
        if first_word == "opaque":
            pending = ""
            continue

        # The challenge data normally has exactly one statement on each line.
        # This fast path matters for multi-million-gate circuits.
        if (
            not pending
            and "{" not in stripped
            and "}" not in stripped
            and stripped.endswith(";")
            and stripped.count(";") == 1
        ):
            yield stripped[:-1].strip()
            continue

        # Braces belonging to ordinary QASM 3 control flow are separators.
        line = line.replace("{", " ").replace("}", " ")
        pending = f"{pending} {line}" if pending else line
        while True:
            semi = pending.find(";")
            if semi < 0:
                break
            statement = pending[:semi].strip()
            pending = pending[semi + 1 :]
            if statement:
                yield statement

    if pending.strip():
        # Helpful for permissive parsing of a final statement with no semicolon.
        yield pending.strip()


def _connected_components(adjacency):
    remaining = set(adjacency)
    count = 0
    while remaining:
        count += 1
        stack = [remaining.pop()]
        while stack:
            node = stack.pop()
            unseen = adjacency[node] & remaining
            if unseen:
                remaining.difference_update(unseen)
                stack.extend(unseen)
    return count


def _min_degree_treewidth(adjacency):
    """Return a min-degree upper bound; exact treewidth is NP-hard."""
    graph = {node: set(neighbors) for node, neighbors in adjacency.items()}
    width = 0
    while graph:
        node = min(graph, key=lambda item: len(graph[item]))
        neighbors = graph[node]
        width = max(width, len(neighbors))
        for left, right in combinations(neighbors, 2):
            graph[left].add(right)
            graph[right].add(left)
        for neighbor in neighbors:
            graph[neighbor].discard(node)
        del graph[node]
    return width


def _normalize_gate_name(name):
    """Collapse automatically numbered custom gates into stable categories."""
    if _MCPHASE_VARIANT_RE.fullmatch(name):
        return "mcphase"
    if _GENERATED_CIRCUIT_GATE_RE.fullmatch(name):
        return "custom_circuit"
    return name


class CircuitFeatureParser:
    def __init__(self, model_features_only=True):
        self.model_features_only = model_features_only

    def _select_features(self, features):
        """Return the model input vector or the complete diagnostic vector."""
        if not self.model_features_only:
            return features

        selected = {
            name: value
            for name, value in features.items()
            if name not in MODEL_EXCLUDED_FEATURES
        }
        selected["inactive_qubits"] = max(
            0, selected["n_qubits"] - selected["active_qubits"]
        )
        return selected

    @staticmethod
    def _resolve_operand(token, registers):
        token = token.strip()
        match = _HARDWARE_QUBIT_RE.fullmatch(token)
        if match:
            return [int(match.group(1))]

        match = _REF_RE.fullmatch(token)
        if not match:
            return []
        name, first, last = match.groups()
        if name not in registers:
            return []
        offset, size = registers[name]
        if first is None:
            return list(range(offset, offset + size))
        first = int(first)
        if last is None:
            return [offset + first] if first < size else []
        last = int(last)
        step = 1 if last >= first else -1
        return [
            offset + index
            for index in range(first, last + step, step)
            if 0 <= index < size
        ]

    # ------------------------------------------------------------------ #
    # 1) FEATURE PARSER: QASM text -> one feature dictionary              #
    # ------------------------------------------------------------------ #
    def featurize(self, qasm_text: str) -> dict:
        if len(qasm_text) > LARGE_QASM_BYTES:
            return self._select_features(self._featurize_large(qasm_text))

        registers = {}
        n_qubits = 0
        n_clbits = 0
        last_layer = []

        gate_counts = defaultdict(int)
        interaction_weights = defaultdict(int)
        per_qubit_ops = []
        operand_cache = {}

        n_ops = 0
        n_1q = 0
        n_2q = 0
        n_multiq = 0
        max_gate_arity = 0
        measurement_count = 0
        reset_count = 0
        barrier_count = 0
        delay_count = 0
        parameterized_gate_count = 0
        diagonal_gate_count = 0
        rotation_gate_count = 0
        clifford_gate_count = 0
        t_gate_count = 0
        s_gate_count = 0

        def ensure_qubits(count):
            nonlocal n_qubits
            if count > n_qubits:
                growth = count - n_qubits
                last_layer.extend([0] * growth)
                per_qubit_ops.extend([0] * growth)
                n_qubits = count

        def add_register(name, size):
            nonlocal n_qubits
            if name in registers:
                return
            registers[name] = (n_qubits, size)
            ensure_qubits(n_qubits + size)

        def resolve_operand(token):
            token = token.strip()
            cached = operand_cache.get(token)
            if cached is not None:
                return cached
            resolved = self._resolve_operand(token, registers)
            operand_cache[token] = resolved
            return resolved

        def record_operation(gate_name, qubits, has_parameters):
            nonlocal n_ops, n_1q, n_2q, n_multiq, max_gate_arity
            nonlocal parameterized_gate_count, diagonal_gate_count
            nonlocal rotation_gate_count, clifford_gate_count
            nonlocal t_gate_count, s_gate_count

            if not qubits:
                return
            if len(qubits) == 2:
                if qubits[0] == qubits[1]:
                    qubits = qubits[:1]
                elif qubits[0] > qubits[1]:
                    qubits = [qubits[1], qubits[0]]
            elif len(qubits) > 2:
                qubits = sorted(set(qubits))
            ensure_qubits(max(qubits) + 1)
            arity = len(qubits)
            n_ops += 1
            gate_counts[gate_name] += 1
            max_gate_arity = max(max_gate_arity, arity)
            if arity == 1:
                n_1q += 1
            elif arity == 2:
                n_2q += 1
            else:
                n_multiq += 1

            if has_parameters:
                parameterized_gate_count += 1
            if gate_name in _DIAGONAL_GATES:
                diagonal_gate_count += 1
            if gate_name in _ROTATION_GATES:
                rotation_gate_count += 1
            if gate_name in _CLIFFORD_GATES:
                clifford_gate_count += 1
            if gate_name in {"t", "tdg"}:
                t_gate_count += 1
            if gate_name in {"s", "sdg"}:
                s_gate_count += 1

            if arity == 1:
                index = qubits[0]
                layer = last_layer[index] + 1
                last_layer[index] = layer
                per_qubit_ops[index] += 1
            elif arity == 2:
                left, right = qubits
                layer = 1 + max(last_layer[left], last_layer[right])
                last_layer[left] = layer
                last_layer[right] = layer
                per_qubit_ops[left] += 1
                per_qubit_ops[right] += 1
                interaction_weights[(left, right)] += 1
            else:
                layer = 1 + max(last_layer[index] for index in qubits)
                for index in qubits:
                    last_layer[index] = layer
                    per_qubit_ops[index] += 1
                for left, right in combinations(qubits, 2):
                    interaction_weights[(left, right)] += 1

        for statement in _iter_statements(qasm_text):
            lowered = statement.lower().strip()

            if lowered.startswith("qreg "):
                match = _QREG2_RE.fullmatch(statement)
                if match:
                    add_register(match.group(1), int(match.group(2)))
                    continue
            elif lowered.startswith("qubit"):
                match = _QUBIT3_ARRAY_RE.fullmatch(statement)
                if match:
                    add_register(match.group(2), int(match.group(1)))
                    continue
                match = _QUBIT3_ALT_RE.fullmatch(statement)
                if match:
                    add_register(match.group(1), int(match.group(2)))
                    continue
                match = _QUBIT3_SCALAR_RE.fullmatch(statement)
                if match:
                    add_register(match.group(1), 1)
                    continue
            elif lowered.startswith("creg "):
                match = _CREG2_RE.fullmatch(statement)
                if match:
                    n_clbits += int(match.group(2))
                    continue
            elif lowered.startswith("bit"):
                match = _BIT3_ARRAY_RE.fullmatch(statement)
                if match:
                    n_clbits += int(match.group(1))
                    continue
                match = _BIT3_SCALAR_RE.fullmatch(statement)
                if match:
                    n_clbits += 1
                    continue

            if lowered.startswith(_SKIP_PREFIXES):
                continue

            # QASM 3 assignment and QASM 2 arrow forms both contain "measure".
            if "measure" in lowered:
                measurement_count += 1
                continue
            if lowered.startswith("barrier"):
                barrier_count += 1
                operand_text = statement[len("barrier") :]
                groups = [
                    resolve_operand(part)
                    for part in operand_text.split(",")
                ]
                qubits = sorted({q for group in groups for q in group})
                if qubits:
                    layer = max(last_layer[q] for q in qubits)
                    for q in qubits:
                        last_layer[q] = layer
                continue
            if lowered.startswith("delay"):
                delay_count += 1
                continue

            statement = _LEADING_IF_RE.sub("", statement).strip()
            # In QASM 3 modifiers such as "ctrl @ inv @ x", the actual gate
            # name and operands occur after the last @.
            if "@" in statement:
                statement = statement.rsplit("@", 1)[1].strip()

            parameter_start = statement.find("(")
            first_space = statement.find(" ")
            if parameter_start >= 0 and (
                first_space < 0 or parameter_start < first_space
            ):
                parameter_end = statement.rfind(")")
                if parameter_end < parameter_start:
                    continue
                gate_name = statement[:parameter_start].strip().lower()
                has_parameters = True
                operand_text = statement[parameter_end + 1 :].strip()
            elif first_space > 0:
                gate_name = statement[:first_space].lower()
                has_parameters = False
                operand_text = statement[first_space + 1 :].strip()
            else:
                match = _OP_RE.fullmatch(statement)
                if not match:
                    continue
                gate_name = match.group(1).lower()
                has_parameters = bool(match.group(2))
                operand_text = match.group(3).strip()

            if gate_name == "reset":
                reset_count += 1
            if "->" in operand_text:
                operand_text = operand_text.split("->", 1)[0]

            groups = [
                resolve_operand(part)
                for part in operand_text.split(",")
            ]
            groups = [group for group in groups if group]
            if not groups:
                continue

            gate_name = _normalize_gate_name(gate_name)

            # OpenQASM register broadcasting applies the gate elementwise.
            applications = max(len(group) for group in groups)
            compatible = all(len(group) in (1, applications) for group in groups)
            if compatible:
                for index in range(applications):
                    qubits = [
                        group[index] if len(group) > 1 else group[0]
                        for group in groups
                    ]
                    record_operation(gate_name, qubits, has_parameters)
            else:
                # Preserve signal from unusual slices rather than failing the
                # whole circuit.
                record_operation(
                    gate_name,
                    [q for group in groups for q in group],
                    has_parameters,
                )

        depth = max(last_layer, default=0)
        active_qubits = sum(count > 0 for count in per_qubit_ops)
        max_qubit_load = max(per_qubit_ops, default=0)
        avg_qubit_load = sum(per_qubit_ops) / n_qubits if n_qubits else 0.0

        adjacency = {index: set() for index in range(n_qubits)}
        weighted_degree = [0] * n_qubits
        weighted_distance = 0
        pair_interactions = 0
        max_2q_dist = 0
        cut_deltas = [0] * (n_qubits + 1)
        for (left, right), weight in interaction_weights.items():
            adjacency[left].add(right)
            adjacency[right].add(left)
            weighted_degree[left] += weight
            weighted_degree[right] += weight
            distance = right - left
            weighted_distance += distance * weight
            pair_interactions += weight
            max_2q_dist = max(max_2q_dist, distance)
            cut_deltas[left] += weight
            cut_deltas[right] -= weight

        running_cut = 0
        max_cutwidth = 0
        for index in range(max(0, n_qubits - 1)):
            running_cut += cut_deltas[index]
            max_cutwidth = max(max_cutwidth, running_cut)

        unique_interactions = len(interaction_weights)
        possible_edges = n_qubits * (n_qubits - 1) / 2
        graph_density = (
            unique_interactions / possible_edges if possible_edges else 0.0
        )
        component_count = _connected_components(adjacency) if adjacency else 0
        treewidth = _min_degree_treewidth(adjacency) if adjacency else 0

        features = {
            "qasm_version": (
                3 if re.search(r"OPENQASM\s+3", qasm_text[:200], re.I) else 2
            ),
            "n_qubits": n_qubits,
            "n_clbits": n_clbits,
            "n_ops": n_ops,
            "n_1q": n_1q,
            "n_2q": n_2q,
            "n_multiq": n_multiq,
            "depth": depth,
            "max_gate_arity": max_gate_arity,
            "active_qubits": active_qubits,
            "avg_qubit_load": avg_qubit_load,
            "max_qubit_load": max_qubit_load,
            "two_qubit_gate_density": n_2q / n_ops if n_ops else 0.0,
            "entangling_gate_density": (
                (n_2q + n_multiq) / n_ops if n_ops else 0.0
            ),
            "parallelism": n_ops / depth if depth else 0.0,
            "unique_interactions": unique_interactions,
            "graph_density": graph_density,
            "connected_components": component_count,
            "treewidth": treewidth,
            "avg_2q_dist": (
                weighted_distance / pair_interactions if pair_interactions else 0.0
            ),
            "max_2q_dist": max_2q_dist,
            "max_cutwidth": max_cutwidth,
            "max_weighted_degree": max(weighted_degree, default=0),
            "measurement_count": measurement_count,
            "reset_count": reset_count,
            "barrier_count": barrier_count,
            "delay_count": delay_count,
            "parameterized_gate_count": parameterized_gate_count,
            "diagonal_gate_count": diagonal_gate_count,
            "rotation_gate_count": rotation_gate_count,
            "clifford_gate_count": clifford_gate_count,
            "t_gate_count": t_gate_count,
            "s_gate_count": s_gate_count,
            "gate_counts": dict(gate_counts),
            "sampled": False,
            "sample_fraction": 1.0,
            "definitions_skipped": False,
            "source_fraction": 1.0,
        }
        return self._select_features(features)

    def _featurize_large(self, qasm_text):
        """Sample huge files so parsing remains safely below the 15 s cap.

        The sample includes the header plus windows throughout the circuit.
        Count-like features are extrapolated using the file's exact semicolon
        count. Ratios and interaction-topology features come from the sample.
        """
        size = len(qasm_text)

        # Some generated QASM 3 files place millions of lines of custom gate
        # definitions before the declarations and executable circuit. A
        # disconnected sample can begin inside a definition and lose its
        # matching brace. Parse the complete executable tail when it is small
        # enough to remain within the time limit.
        declaration = _DECLARATION_LINE_RE.search(qasm_text)
        if declaration and declaration.start() > SAMPLE_CHUNK_BYTES:
            version = 3 if re.search(r"OPENQASM\s+3", qasm_text[:200], re.I) else 2
            executable_text = (
                f"OPENQASM {version}.0;\n" + qasm_text[declaration.start() :]
            )
            if len(executable_text) <= LARGE_QASM_BYTES:
                features = self.featurize(executable_text)
                features["definitions_skipped"] = True
                features["source_fraction"] = len(executable_text) / size
                return features

        def complete_chunk(start, length):
            if start > 0:
                start = qasm_text.find("\n", start)
                if start < 0:
                    return ""
                start += 1
            end = min(size, start + length)
            if end < size:
                line_end = qasm_text.find("\n", end)
                end = size if line_end < 0 else line_end + 1
            return qasm_text[start:end]

        starts = (0, size // 3, (2 * size) // 3, max(0, size - SAMPLE_CHUNK_BYTES))
        chunks = [complete_chunk(start, SAMPLE_CHUNK_BYTES) for start in starts]
        sample_text = "\n".join(chunk for chunk in chunks if chunk)
        features = self.featurize(sample_text)

        full_statements = qasm_text.count(";")
        sample_statements = max(sample_text.count(";"), 1)
        sample_ops = features["n_ops"]
        # Header/declaration statements are captured in the first window and
        # terminal measurements are captured in the last one.
        sampled_non_ops = max(0, sample_statements - sample_ops)
        estimated_ops = max(0, full_statements - sampled_non_ops)
        scale = estimated_ops / sample_ops if sample_ops else 1.0

        integer_counts = (
            "n_1q", "n_2q", "n_multiq", "depth",
            "max_qubit_load", "max_cutwidth", "max_weighted_degree",
            "parameterized_gate_count", "diagonal_gate_count",
            "rotation_gate_count", "clifford_gate_count", "t_gate_count",
            "s_gate_count",
        )
        for key in integer_counts:
            features[key] = int(round(features[key] * scale))
        features["n_ops"] = estimated_ops
        features["avg_qubit_load"] *= scale
        features["gate_counts"] = {
            gate: int(round(count * scale))
            for gate, count in features["gate_counts"].items()
        }

        n_ops = features["n_ops"]
        features["two_qubit_gate_density"] = (
            features["n_2q"] / n_ops if n_ops else 0.0
        )
        features["entangling_gate_density"] = (
            (features["n_2q"] + features["n_multiq"]) / n_ops
            if n_ops else 0.0
        )
        features["parallelism"] = (
            n_ops / features["depth"] if features["depth"] else 0.0
        )
        features["sampled"] = True
        features["sample_fraction"] = len(sample_text) / size
        features["definitions_skipped"] = False
        features["source_fraction"] = 1.0
        return features

In [75]:
import json
from pathlib import Path


parser = CircuitFeatureParser(model_features_only=True)


def read_qasm(path):
    """Read a .qasm or decompress a .qasm.zst file."""
    path = Path(path)

    if path.name.endswith(".qasm.zst"):
        import zstandard as zstd

        with path.open("rb") as source_file:
            with zstd.ZstdDecompressor().stream_reader(source_file) as reader:
                return reader.read().decode("utf-8", "replace")

    return path.read_text(encoding="utf-8", errors="replace")


def extract_directory(circuits_dir, output_path=None):
    """Return one feature dictionary for every circuit."""
    circuits_dir = Path(circuits_dir)

    paths = sorted(
        path
        for path in circuits_dir.rglob("*")
        if path.is_file()
        and (
            path.name.endswith(".qasm")
            or path.name.endswith(".qasm.zst")
        )
        and "-checkpoint" not in path.name
        and ".ipynb_checkpoints" not in path.parts
    )

    features_by_circuit = {}

    for index, path in enumerate(paths, 1):
        if path.name.endswith(".qasm.zst"):
            filename = path.name[:-4]  # Remove only ".zst"
        else:
            filename = path.name

        if filename in features_by_circuit:
            print(f"Skipping duplicate circuit: {filename}")
            continue

        qasm_text = read_qasm(path)
        features_by_circuit[filename] = parser.featurize(qasm_text)

        if index % 25 == 0 or index == len(paths):
            print(f"Processed {index}/{len(paths)} circuits")

    if output_path is not None:
        output_path = Path(output_path)
        output_path.write_text(
            json.dumps(features_by_circuit, indent=2),
            encoding="utf-8",
        )
        print(
            f"Saved {len(features_by_circuit)} feature vectors "
            f"to {output_path}"
        )

    return features_by_circuit


features_by_circuit = extract_directory(
    "partial-circuits-360",
    "circuit_features_360_ready.json",
)

Processed 25/360 circuits
Processed 50/360 circuits
Processed 75/360 circuits
Processed 100/360 circuits
Processed 125/360 circuits
Processed 150/360 circuits
Processed 175/360 circuits
Processed 200/360 circuits
Processed 225/360 circuits
Processed 250/360 circuits
Processed 275/360 circuits
Processed 300/360 circuits
Processed 325/360 circuits
Processed 350/360 circuits
Processed 360/360 circuits
Saved 360 feature vectors to circuit_features_360_ready.json


In [76]:
import pandas as pd
import json

# 1. Load the JSON data
with open("circuit_features_360_ready.json", "r") as f:
    data = json.load(f)

# 2. Convert to a DataFrame (rows = files, cols = features)
df = pd.DataFrame.from_dict(data, orient='index')

# 3. Flatten the 'gate_counts' dictionaries into their own separate DataFrame
# This creates a column for every unique gate and fills missing ones with 0.0
gate_df = pd.json_normalize(df['gate_counts']).fillna(0)
gate_df.index = df.index

# Add a prefix so you know these columns represent gate counts
gate_df = gate_df.add_prefix('gate_count_')

# 4. Drop the original nested dictionary and join the new flattened columns
df = df.drop('gate_counts', axis=1).join(gate_df)

# Now your data is perfectly 2D and ready for ML algorithms
df.to_json("circuit_features_partial_360-flattened.json", orient="index", indent=2)
df.to_csv("circuit_features_partial_360-flattened.csv", index=True, index_label="filename")

In [78]:
df.to_json("circuit_features_360-flattened.json", orient="index", indent=2)
print("Successfully saved to circuit_features_flattened.json")

Successfully saved to circuit_features_flattened.json


In [80]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


CAP_SECONDS = 14_400.0
N_SPLITS = 5
RANDOM_STATE = 42


def resolve_input(filename):
    """Find an input beside the notebook or in an upload/ subdirectory."""
    candidates = [Path(filename), Path("upload") / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Put it beside this notebook or in upload/."
    )


FEATURES_PATH = resolve_input("circuit_features_partial_360-flattened.json")
LABELS_PATH = resolve_input("runtime-data-360.csv")
ARTIFACTS_DIR = Path("xgb_artifacts")

print("Features:", FEATURES_PATH)
print("Labels:  ", LABELS_PATH)


Features: circuit_features_partial_360-flattened.json
Labels:   runtime-data-360.csv


In [81]:
with FEATURES_PATH.open(encoding="utf-8") as source:
    feature_map = json.load(source)

if not isinstance(feature_map, dict) or not feature_map:
    raise ValueError("The feature JSON must be a non-empty dictionary.")

first_vector = next(iter(feature_map.values()))
base_feature_names = [
    name for name in first_vector
    if name != "gate_counts"
]
gate_names = sorted({
    gate_name
    for vector in feature_map.values()
    for gate_name in vector.get("gate_counts", {})
})
gate_feature_names = [f"gate_count__{name}" for name in gate_names]

feature_rows = []
for filename, vector in feature_map.items():
    row = {"filename": filename}
    for name in base_feature_names:
        row[name] = vector[name]
    gate_counts = vector.get("gate_counts", {})
    for gate_name, column_name in zip(gate_names, gate_feature_names):
        row[column_name] = gate_counts.get(gate_name, 0)
    feature_rows.append(row)

feature_df = pd.DataFrame(feature_rows)

expected_columns = 1 + len(base_feature_names) + len(gate_feature_names)
if feature_df.shape[1] != expected_columns:
    raise AssertionError("Unexpected number of flattened feature columns.")
if feature_df["filename"].duplicated().any():
    raise ValueError("Duplicate filenames found in the feature JSON.")

print(f"Circuits:              {len(feature_df):,}")
print(f"Fixed features:        {len(base_feature_names):,}")
print(f"Gate-count features:   {len(gate_feature_names):,}")
print(f"Total circuit inputs:  {len(base_feature_names) + len(gate_feature_names):,}")
feature_df.head()


Circuits:              360
Fixed features:        92
Gate-count features:   0
Total circuit inputs:  92


,filename,n_qubits,n_ops,n_1q,n_2q,n_multiq,depth,max_gate_arity,active_qubits,avg_qubit_load,...,gate_count_y,gate_count_u,gate_count_crx,gate_count_rzx,gate_count_xx_minus_yy,gate_count_cu3,gate_count_iswap,gate_count_xx_plus_yy,gate_count_csx,gate_count_add4
0,0071a13b.qasm,47,14537,12143,2394,0,502,2,47,360.234043,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,03042ac9.qasm,47,14537,12143,2394,0,502,2,47,360.234043,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,03934028.qasm,128,655180,326170,189876,139134,361696,3,128,8775.968750,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,04276362.qasm,3,33,23,10,0,21,2,3,14.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0491d461.qasm,16,4400,3680,720,0,500,2,16,320.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [82]:
labels = pd.read_csv(LABELS_PATH)
required_label_columns = {"filename", "threshold", "duration_s", "status"}
missing_label_columns = required_label_columns - set(labels.columns)
if missing_label_columns:
    raise ValueError(f"Missing label columns: {sorted(missing_label_columns)}")

labels["filename"] = labels["filename"].astype(str).str.strip()
labels["threshold"] = pd.to_numeric(labels["threshold"], errors="raise").astype(int)
labels["status"] = labels["status"].astype(str).str.strip().str.lower()
labels["is_timeout"] = labels["status"].eq("timeout")
labels["duration_s"] = pd.to_numeric(labels["duration_s"], errors="coerce")

bad_success = labels["status"].eq("success") & labels["duration_s"].isna()
if bad_success.any():
    bad_files = labels.loc[bad_success, "filename"].tolist()[:10]
    raise ValueError(f"Successful rows with missing duration: {bad_files}")

labels["target_duration_s"] = np.where(
    labels["is_timeout"],
    CAP_SECONDS,
    labels["duration_s"],
).astype(float)
labels["target_log1p"] = np.log1p(labels["target_duration_s"])

data = labels.merge(feature_df, on="filename", how="left", validate="many_to_one")
missing_features = data["n_qubits"].isna()
if missing_features.any():
    missing_files = data.loc[missing_features, "filename"].unique().tolist()[:10]
    raise ValueError(f"Labels without feature vectors: {missing_files}")

feature_columns = base_feature_names + gate_feature_names + ["threshold"]
X = data[feature_columns].astype(np.float32)
y_log = data["target_log1p"].astype(np.float32)
y_seconds = data["target_duration_s"].to_numpy(dtype=float)
groups = data["filename"].astype(str)
timeout_mask = data["is_timeout"].to_numpy(dtype=bool)

if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Non-finite values found in X.")
if not np.isfinite(y_log.to_numpy()).all():
    raise ValueError("Non-finite values found in y.")
if data[["filename", "threshold"]].duplicated().any():
    raise ValueError("Duplicate (filename, threshold) rows found.")

print(f"Training rows:         {len(data):,}")
print(f"Unique circuits:       {groups.nunique():,}")
print(f"Model input columns:   {len(feature_columns):,}")
print(f"Timeout rows:          {timeout_mask.sum():,}")
print("Threshold counts:")
print(data["threshold"].value_counts().sort_index())


Training rows:         1,016
Unique circuits:       360
Model input columns:   93
Timeout rows:          22
Threshold counts:
threshold
16     360
64     358
512    298
Name: count, dtype: int64


In [83]:
def competition_scores(actual_seconds, predicted_seconds, is_timeout):
    actual = np.asarray(actual_seconds, dtype=float)
    predicted = np.maximum(np.asarray(predicted_seconds, dtype=float), 1e-9)
    is_timeout = np.asarray(is_timeout, dtype=bool)

    scored_prediction = predicted.copy()
    scored_prediction[is_timeout] = np.minimum(
        scored_prediction[is_timeout], CAP_SECONDS
    )

    scores = np.maximum(
        0.0,
        1.0 - np.abs(np.log10(scored_prediction / actual)) / 2.0,
    )
    return scores


def summarize_predictions(actual_seconds, predicted_seconds, is_timeout):
    actual = np.asarray(actual_seconds, dtype=float)
    predicted = np.maximum(np.asarray(predicted_seconds, dtype=float), 1e-9)
    scores = competition_scores(actual, predicted, is_timeout)
    return {
        "competition_score": float(scores.mean()),
        "median_score": float(np.median(scores)),
        "mae_log1p": float(
            np.mean(np.abs(np.log1p(predicted) - np.log1p(actual)))
        ),
        "median_ratio": float(np.median(predicted / actual)),
    }


In [84]:
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is not installed. Run the installation cell near the top."
    ) from exc

print("XGBoost version:", xgb.__version__)

XGB_PARAMS = {
    "objective": "reg:squarederror",
    "n_estimators": 1_200,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 3,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 3.0,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

XGB_PARAMS


XGBoost version: 3.4.1


{'objective': 'reg:squarederror',
 'n_estimators': 1200,
 'learning_rate': 0.03,
 'max_depth': 4,
 'min_child_weight': 3,
 'subsample': 0.85,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.1,
 'reg_lambda': 3.0,
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1}

In [85]:
group_cv = GroupKFold(n_splits=N_SPLITS)
oof_predictions = np.full(len(data), np.nan, dtype=float)
fold_results = []

for fold, (train_idx, valid_idx) in enumerate(
    group_cv.split(X, y_log, groups=groups),
    start=1,
):
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X.iloc[train_idx], y_log.iloc[train_idx], verbose=False)

    predicted_log = model.predict(X.iloc[valid_idx])
    predicted_seconds = np.maximum(np.expm1(predicted_log), 1e-6)
    oof_predictions[valid_idx] = predicted_seconds

    metrics = summarize_predictions(
        y_seconds[valid_idx],
        predicted_seconds,
        timeout_mask[valid_idx],
    )
    metrics.update({
        "fold": fold,
        "train_circuits": groups.iloc[train_idx].nunique(),
        "valid_circuits": groups.iloc[valid_idx].nunique(),
        "valid_rows": len(valid_idx),
    })
    fold_results.append(metrics)
    print(
        f"Fold {fold}: score={metrics['competition_score']:.4f}, "
        f"median score={metrics['median_score']:.4f}, "
        f"log MAE={metrics['mae_log1p']:.4f}"
    )

if np.isnan(oof_predictions).any():
    raise AssertionError("Some rows did not receive an out-of-fold prediction.")

fold_report = pd.DataFrame(fold_results).set_index("fold")
overall_metrics = summarize_predictions(
    y_seconds,
    oof_predictions,
    timeout_mask,
)

print("\nFold results")
display(fold_report)
print("Overall out-of-fold metrics")
display(pd.Series(overall_metrics, name="OOF"))


Fold 1: score=0.8191, median score=0.9222, log MAE=0.5528
Fold 2: score=0.8503, median score=0.9309, log MAE=0.4423
Fold 3: score=0.8805, median score=0.9529, log MAE=0.3384
Fold 4: score=0.8135, median score=0.9100, log MAE=0.5841
Fold 5: score=0.8735, median score=0.9616, log MAE=0.3956

Fold results


,competition_score,median_score,mae_log1p,median_ratio,train_circuits,valid_circuits,valid_rows
fold,,,,,,,
1,0.819071,0.922197,0.552792,1.007210,288,72,204
2,0.850277,0.930928,0.442335,1.002454,288,72,203
3,0.880463,0.952913,0.338432,0.979809,288,72,203
4,0.813471,0.909981,0.584092,1.047031,288,72,203
5,0.873526,0.961569,0.395559,0.998424,288,72,203


Overall out-of-fold metrics


competition_score    0.847334
median_score         0.937557
mae_log1p            0.462731
median_ratio         1.001538
Name: OOF, dtype: float64

In [86]:
oof_report = data[[
    "filename",
    "threshold",
    "status",
    "target_duration_s",
]].copy()
oof_report["predicted_duration_s"] = oof_predictions
oof_report["score"] = competition_scores(
    y_seconds,
    oof_predictions,
    timeout_mask,
)
oof_report["prediction_ratio"] = oof_predictions / y_seconds

summary_by_threshold = oof_report.groupby("threshold").agg(
    rows=("score", "size"),
    mean_score=("score", "mean"),
    median_score=("score", "median"),
    median_ratio=("prediction_ratio", "median"),
)

summary_by_status = oof_report.groupby("status").agg(
    rows=("score", "size"),
    mean_score=("score", "mean"),
    median_score=("score", "median"),
    median_ratio=("prediction_ratio", "median"),
)

display(summary_by_threshold)
display(summary_by_status)
display(oof_report.nsmallest(20, "score"))


,rows,mean_score,median_score,median_ratio
threshold,,,,
16,360,0.867464,0.948092,1.026964
64,358,0.858184,0.932309,0.987116
512,298,0.809981,0.920976,0.990059


,rows,mean_score,median_score,median_ratio
status,,,,
success,994,0.853583,0.941287,1.004598
timeout,22,0.564965,0.676333,0.227659


,filename,threshold,status,target_duration_s,predicted_duration_s,score,prediction_ratio
19,0534bf07.qasm,512,timeout,14400.000000,7.811742e+01,0.000000,0.005425
80,173d4b8b.qasm,512,success,0.457151,1.000000e-06,0.000000,0.000002
264,65dcd512.qasm,64,success,0.517777,1.000000e-06,0.000000,0.000002
288,68528bdf.qasm,64,success,0.166059,1.000000e-06,0.000000,0.000006
319,72b5f996.qasm,512,success,0.175735,1.168293e+02,0.000000,664.802732
338,749ea355.qasm,16,success,0.229153,2.938735e+02,0.000000,1282.431779
340,749ea355.qasm,512,success,0.222562,8.500516e+02,0.000000,3819.395942
356,758554e0.qasm,16,success,755.028116,6.984257e+00,0.000000,0.009250
357,758554e0.qasm,64,success,590.694655,5.102612e+00,0.000000,0.008638
389,827a3442.qasm,512,success,0.188646,1.000000e-06,0.000000,0.000005


In [87]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

final_model = XGBRegressor(**XGB_PARAMS)
final_model.fit(X, y_log, verbose=False)

model_path = ARTIFACTS_DIR / "xgb_runtime_model.json"
columns_path = ARTIFACTS_DIR / "feature_columns.json"
metadata_path = ARTIFACTS_DIR / "training_metadata.json"
oof_path = ARTIFACTS_DIR / "oof_predictions.csv"

final_model.save_model(model_path)
columns_path.write_text(
    json.dumps(feature_columns, indent=2),
    encoding="utf-8",
)
metadata_path.write_text(
    json.dumps(
        {
            "target": "log1p(runtime_seconds)",
            "timeout_target_seconds": CAP_SECONDS,
            "training_rows": len(data),
            "training_circuits": int(groups.nunique()),
            "feature_count": len(feature_columns),
            "xgboost_version": xgb.__version__,
            "parameters": XGB_PARAMS,
            "oof_metrics": overall_metrics,
        },
        indent=2,
    ),
    encoding="utf-8",
)
oof_report.to_csv(oof_path, index=False)

print("Saved:")
print(" ", model_path)
print(" ", columns_path)
print(" ", metadata_path)
print(" ", oof_path)


Saved:
  xgb_artifacts/xgb_runtime_model.json
  xgb_artifacts/feature_columns.json
  xgb_artifacts/training_metadata.json
  xgb_artifacts/oof_predictions.csv


In [88]:
importance = pd.Series(
    final_model.feature_importances_,
    index=feature_columns,
    name="importance",
).sort_values(ascending=False)

display(importance.head(94).to_frame())


,importance
max_cutwidth,0.140792
n_2q,0.119652
n_ops,0.051871
rotation_gate_count,0.051465
gate_count_custom_circuit,0.050345
...,...
gate_count_ch,0.000000
gate_count_cu,0.000000
gate_count_dcx,0.000000
gate_count_c3sx,0.000000


In [89]:
# 1. Create your DataFrame for the top 94
top_94 = importance.head(94).to_frame()

# 2. Display ALL 94 rows without truncation
with pd.option_context('display.max_rows', 100):
    display(top_94)

# 3. Save them to a CSV file in your current working directory
top_94.to_csv("feature_importances.csv", index_label="feature")

,importance
max_cutwidth,0.140792
n_2q,0.119652
n_ops,0.051871
rotation_gate_count,0.051465
gate_count_custom_circuit,0.050345
max_weighted_degree,0.047517
parallelism,0.044330
gate_count_q,0.035411
treewidth,0.033136
threshold,0.031965


In [62]:
def flatten_one_circuit(feature_vector, threshold, columns):
    row = {
        name: value
        for name, value in feature_vector.items()
        if name != "gate_counts"
    }
    for gate_name, count in feature_vector.get("gate_counts", {}).items():
        row[f"gate_count__{gate_name}"] = count
    row["threshold"] = int(threshold)

    # Reindex creates any missing gate columns and keeps training order.
    frame = pd.DataFrame([row]).reindex(columns=columns, fill_value=0)
    return frame.astype(np.float32)


def predict_runtime_seconds(model, feature_vector, threshold, columns):
    model_input = flatten_one_circuit(feature_vector, threshold, columns)
    predicted_log = float(model.predict(model_input)[0])
    return float(max(np.expm1(predicted_log), 1e-6))


# Sanity check on one known circuit. This is only an inference check, not a
# validation score because the final model was trained on all labeled rows.
example_filename = next(iter(feature_map))
example_threshold = 400
example_prediction = predict_runtime_seconds(
    final_model,
    feature_map[example_filename],
    example_threshold,
    feature_columns,
)
print(example_filename, example_threshold, example_prediction)

0071a13b.qasm 400 11.204928873494943


In [63]:
def predict_runtime(filename, threshold):
    if threshold not in {16, 64, 512}:
        raise ValueError("threshold must be 16, 64, or 512")

    matching = feature_df[
        feature_df["filename"].astype(str).str.strip() == filename.strip()
    ]

    if matching.empty:
        raise ValueError(f"No feature vector found for {filename}")

    circuit_features = matching.iloc[0].to_dict()
    circuit_features["threshold"] = threshold

    # Build columns in exactly the same order used during training
    input_row = pd.DataFrame([{
        column: circuit_features.get(column, 0.0)
        for column in feature_columns
    }])

    input_row = input_row[feature_columns].astype(np.float32)

    predicted_log = final_model.predict(input_row)[0]
    predicted_seconds = float(np.expm1(predicted_log))

    return max(predicted_seconds, 0.0)

In [65]:
filename = "0071a13b.qasm"

for threshold in [16, 64, 512]:
    prediction = predict_runtime(filename, threshold)
    print(filename, threshold, prediction)

0071a13b.qasm 16 1.9585890769958496
0071a13b.qasm 64 11.20492935180664
0071a13b.qasm 512 477.7330017089844


In [66]:
def predict_new_qasm(qasm_path, threshold, parser, model, feature_columns):
    if threshold not in {16, 64, 512}:
        raise ValueError("Threshold must be 16, 64, or 512.")

    # Read and featurize the new circuit
    qasm_text = read_qasm(qasm_path)
    features = parser.featurize(qasm_text)

    # Flatten nested gate counts, if still nested
    flat_features = {
        key: value
        for key, value in features.items()
        if key != "gate_counts"
    }

    for gate, count in features.get("gate_counts", {}).items():
        flat_features[f"gate_count__{gate}"] = count

    # Threshold is a model input
    flat_features["threshold"] = threshold

    # Missing gate types become zero.
    # Extra gates not used during training are ignored.
    input_row = pd.DataFrame([{
        column: flat_features.get(column, 0.0)
        for column in feature_columns
    }])

    input_row = input_row[feature_columns].astype(np.float32)

    predicted_log = model.predict(input_row)[0]
    predicted_seconds = float(np.expm1(predicted_log))

    return max(predicted_seconds, 0.0)

In [69]:
new_qasm_path = "extracted_circuits/deb00fe1.qasm"

for threshold in [16, 64, 512]:
    prediction = predict_new_qasm(
        qasm_path=new_qasm_path,
        threshold=threshold,
        parser=parser,
        model=final_model,
        feature_columns=feature_columns,
    )

    print(
        Path(new_qasm_path).name,
        threshold,
        prediction,
    )

deb00fe1.qasm 16 0.16154438257217407
deb00fe1.qasm 64 0.4048178493976593
deb00fe1.qasm 512 0.19623669981956482


In [91]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

TRAIN_FEATURES_PATH = Path("circuit_features_partial_360-flattened.json")
ALL_FEATURES_PATH = Path("circuit_features_ready.json")

with TRAIN_FEATURES_PATH.open("r", encoding="utf-8") as f:
    training_features = json.load(f)

with ALL_FEATURES_PATH.open("r", encoding="utf-8") as f:
    all_features = json.load(f)

training_filenames = set(training_features)
all_filenames = set(all_features)
remaining_filenames = sorted(all_filenames - training_filenames)

print("Training circuits:", len(training_filenames))
print("All circuits:", len(all_filenames))
print("Remaining circuits:", len(remaining_filenames))

assert len(training_filenames) == 360
assert len(all_filenames) == 532
assert len(remaining_filenames) == 172

Training circuits: 360
All circuits: 532
Remaining circuits: 172


In [92]:
def flatten_feature_vector(features):
    """Support nested and already-flattened feature dictionaries."""
    flat = {}

    for key, value in features.items():
        if key == "gate_counts":
            for gate_name, gate_count in value.items():
                flat[f"gate_count__{gate_name}"] = gate_count
        elif isinstance(value, (int, float, np.integer, np.floating)):
            flat[key] = value

    return flat


VALID_THRESHOLDS = [16, 64, 512]
prediction_rows = []

for filename in remaining_filenames:
    flat_features = flatten_feature_vector(all_features[filename])

    for threshold in VALID_THRESHOLDS:
        model_values = {}

        for column in feature_columns:
            if column == "threshold":
                model_values[column] = threshold
            else:
                # Unseen/missing gate categories become zero
                model_values[column] = flat_features.get(column, 0.0)

        input_row = pd.DataFrame(
            [model_values],
            columns=feature_columns,
            dtype=np.float32,
        )

        predicted_log = final_model.predict(input_row)[0]
        predicted_seconds = max(
            0.0,
            float(np.expm1(predicted_log)),
        )

        prediction_rows.append({
            "filename": filename,
            "threshold": threshold,
            "predicted_duration_s": predicted_seconds,
        })

remaining_predictions = pd.DataFrame(prediction_rows)

print("Circuits predicted:", remaining_predictions["filename"].nunique())
print("Prediction rows:", len(remaining_predictions))
print(remaining_predictions.head(177))

Circuits predicted: 172
Prediction rows: 516
          filename  threshold  predicted_duration_s
0    00c20ece.qasm         16              3.496338
1    00c20ece.qasm         64             12.333995
2    00c20ece.qasm        512             23.157639
3    03ee478e.qasm         16              0.293795
4    03ee478e.qasm         64              1.285297
..             ...        ...                   ...
172  31fd025e.qasm         64              7.074141
173  31fd025e.qasm        512            240.580826
174  32de9cc1.qasm         16              1.370622
175  32de9cc1.qasm         64              2.617763
176  32de9cc1.qasm        512             19.091232

[177 rows x 3 columns]


In [94]:
remaining_predictions.to_csv(
    "predictions-remaining-172.csv",
    index=False,
)

print("Saved predictions-remaining-172.csv")

Saved predictions-remaining-172.csv


In [95]:
predicted_filenames = set(remaining_predictions["filename"])

overlap = training_filenames & predicted_filenames
missing = all_filenames - training_filenames - predicted_filenames

print("Training/prediction overlap:", len(overlap))
print("Remaining circuits not predicted:", len(missing))

assert len(overlap) == 0
assert len(missing) == 0

Training/prediction overlap: 0
Remaining circuits not predicted: 0
